# Week 7 Community Contribution - Adnan Gobeljic

This notebook follows the Week 7 flow from the course:

1. QLoRA setup (Day 1)
2. Prompt dataset loading and formatting (Day 2)
3. Fine-tuning with `SFTTrainer` (Days 3 and 4)
4. Evaluation with `util.evaluate()` (Day 5)

Target runtime: Google Colab with a GPU.

## Environment setup

Run this in Colab before importing libraries. If the packages are already installed, you can skip it.

In [1]:
# !pip install -q --upgrade bitsandbytes trl transformers accelerate datasets peft huggingface_hub
# !wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py

## Imports

In [2]:
import os
from datetime import datetime
from urllib.request import urlretrieve

import torch
from datasets import load_dataset
from huggingface_hub import login
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from trl import SFTConfig, SFTTrainer

try:
    from google.colab import userdata
except ImportError:
    userdata = None

## Configuration

In [ ]:
BASE_MODEL_CANDIDATES = [
    "meta-llama/Llama-3.2-3B",   # gated: requires approved HF access
    "Qwen/Qwen2.5-3B-Instruct",  # open fallback
    "Qwen/Qwen2.5-1.5B-Instruct",# lighter fallback
    "Qwen/Qwen2.5-0.5B-Instruct",# lightest fallback for local runs
]

DATA_USER = "ed-donner"
LITE_MODE = True

DATASET_NAME = f"{DATA_USER}/items_prompts_lite" if LITE_MODE else f"{DATA_USER}/items_prompts_full"
HF_USER = os.getenv("HF_USER", "adnangobeljic")
PROJECT_NAME = "price"
RUN_NAME = f"{datetime.now():%Y-%m-%d_%H.%M.%S}-lite" if LITE_MODE else f"{datetime.now():%Y-%m-%d_%H.%M.%S}-full"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

# Training hyperparameters
EPOCHS = 1 if LITE_MODE else 2
BATCH_SIZE = 16 if LITE_MODE else 8
MAX_SEQUENCE_LENGTH = 128
GRADIENT_ACCUMULATION_STEPS = 1
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.01
SAVE_STEPS = 100
LOG_STEPS = 10

LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.1
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]

HAS_CUDA = torch.cuda.is_available()
HAS_MPS = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()

if HAS_CUDA:
    capability = torch.cuda.get_device_capability()
else:
    capability = (0, 0)

use_bf16 = HAS_CUDA and capability[0] >= 8
print(f"Dataset: {DATASET_NAME}")
print(f"Run name: {PROJECT_RUN_NAME}")
print(f"Model candidates: {BASE_MODEL_CANDIDATES}")
print(f"CUDA: {HAS_CUDA}, MPS: {HAS_MPS}, bf16 enabled: {use_bf16}")

In [4]:
hf_token = None
if userdata is not None:
    try:
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None

if not hf_token:
    hf_token = os.getenv("HF_TOKEN")

if hf_token:
    login(hf_token, add_to_git_credential=True)
    print("Logged in to Hugging Face")
else:
    print("HF token not found. Set Colab secret HF_TOKEN or env var HF_TOKEN.")

dataset = load_dataset(DATASET_NAME)
train = dataset["train"]
val = dataset["val"]
test = dataset["test"]

def add_text(example):
    return {"text": example["prompt"] + example["completion"]}

train_sft = train.map(add_text, remove_columns=train.column_names)
val_sft = val.map(add_text, remove_columns=val.column_names)

print(f"Train: {len(train_sft):,}, Val: {len(val_sft):,}, Test: {len(test):,}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in to Hugging Face
Train: 20,000, Val: 1,000, Test: 1,000


## Load base model and fine-tune with QLoRA

The notebook will try model candidates in order. If your account cannot access gated Llama models, it will automatically fall back to `Qwen/Qwen2.5-3B-Instruct`.

In [ ]:
ENABLE_4BIT = False
quant_config = None

if HAS_CUDA:
    try:
        import bitsandbytes as bnb
        print(f"bitsandbytes detected: {bnb.__version__}")
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
            bnb_4bit_quant_type="nf4",
        )
        ENABLE_4BIT = True
    except Exception as e:
        print(f"bitsandbytes unavailable/unsupported, using non-quantized loading: {e}")
else:
    print("CUDA is not available; disabling 4-bit quantization and loading model normally.")


def load_model_with_fallback(model_candidates):
    last_error = None

    for model_name in model_candidates:
        try:
            print(f"Trying model: {model_name}")
            tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            tokenizer.padding_side = "right"

            model_kwargs = {}
            if ENABLE_4BIT:
                model_kwargs["quantization_config"] = quant_config
                model_kwargs["device_map"] = "auto"
            else:
                model_kwargs["torch_dtype"] = torch.float16 if (HAS_CUDA or HAS_MPS) else torch.float32

            base_model = AutoModelForCausalLM.from_pretrained(
                model_name,
                **model_kwargs,
            )

            if not ENABLE_4BIT:
                target_device = "cuda" if HAS_CUDA else ("mps" if HAS_MPS else "cpu")
                base_model = base_model.to(target_device)

            base_model.generation_config.pad_token_id = tokenizer.pad_token_id
            print(f"Loaded model: {model_name}")
            print(f"4-bit quantization enabled: {ENABLE_4BIT}")
            print(f"Base model memory: {base_model.get_memory_footprint() / 1e6:.1f} MB")
            return model_name, tokenizer, base_model
        except Exception as e:
            last_error = e
            msg = str(e).lower()
            if "gated" in msg or "403" in msg or "forbidden" in msg:
                print(f"Access denied for {model_name} (likely gated). Trying next model...")
            else:
                print(f"Failed to load {model_name}: {e}")

    raise RuntimeError(
        "Could not load any base model from BASE_MODEL_CANDIDATES. "
        "If you want Llama, request access on Hugging Face; otherwise keep using the open fallback models."
    ) from last_error


BASE_MODEL, tokenizer, base_model = load_model_with_fallback(BASE_MODEL_CANDIDATES)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

effective_batch_size = BATCH_SIZE if ENABLE_4BIT else min(2, BATCH_SIZE)
effective_grad_accum = GRADIENT_ACCUMULATION_STEPS if ENABLE_4BIT else max(4, GRADIENT_ACCUMULATION_STEPS)

train_args = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=effective_batch_size,
    per_device_eval_batch_size=1 if not ENABLE_4BIT else 2,
    gradient_accumulation_steps=effective_grad_accum,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    fp16=HAS_CUDA and not use_bf16,
    bf16=HAS_CUDA and use_bf16,
    max_grad_norm=0.3,
    logging_steps=LOG_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    max_length=MAX_SEQUENCE_LENGTH,
    dataset_text_field="text",
    report_to="none",
)

print(f"Training batch size: {effective_batch_size}, grad accumulation: {effective_grad_accum}")

trainer = SFTTrainer(
    model=base_model,
    train_dataset=train_sft,
    eval_dataset=val_sft,
    peft_config=lora_config,
    args=train_args,
)

In [ ]:
trainer.train()
fine_tuned_model = trainer.model
print("Training complete")

# Optional: push adapter to your Hugging Face account
# fine_tuned_model.push_to_hub(HUB_MODEL_NAME, private=True)
# tokenizer.push_to_hub(HUB_MODEL_NAME)

## Evaluation (Week 7 Day 5 style)

In [ ]:
if not os.path.exists("util.py"):
    urlretrieve(
        "https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py",
        "util.py",
    )

from util import evaluate

In [ ]:
INFER_DEVICE = "cuda" if HAS_CUDA else ("mps" if HAS_MPS else "cpu")
fine_tuned_model = fine_tuned_model.to(INFER_DEVICE)


def model_predict(item):
    inputs = tokenizer(item["prompt"], return_tensors="pt").to(INFER_DEVICE)
    with torch.no_grad():
        output_ids = fine_tuned_model.generate(**inputs, max_new_tokens=8)
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    return tokenizer.decode(generated_ids)

set_seed(42)
evaluate(model_predict, test)

In [ ]:
def predict_from_description(description):
    prompt = (
        "What does this cost to the nearest dollar?\n\n"
        f"{description}\n\n"
        "Price is $"
    )
    item = {"prompt": prompt}
    return model_predict(item)

example = "Wireless mechanical keyboard with hot-swap switches and RGB backlight"
print(predict_from_description(example))